# Building Multi-Step Tool-Calling Datasets with Data Designer

## A Complete Tutorial for Generating Training Data for Agentic RL

This notebook teaches you how to build synthetic datasets for training multi-step tool-calling agents using NVIDIA's Data Designer. By the end, you'll understand:

1. **Environment Design**: How to structure tools and databases for agentic tasks
2. **Synthetic Data Generation**: Using LLMs to generate realistic user queries and tool trajectories
3. **Quality Assurance**: Using LLM judges to filter and audit generated data
4. **RL Integration**: Formatting data for NeMo Gym rollout collection and GRPO training

---

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         DATA GENERATION PIPELINE                         │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐              │
│  │ Tool Schemas │───▶│ User Query   │───▶│  Trajectory  │              │
│  │   (Seed)     │    │ Generation   │    │  Simulation  │              │
│  └──────────────┘    └──────────────┘    └──────────────┘              │
│                                                 │                        │
│                                                 ▼                        │
│                                          ┌──────────────┐              │
│                                          │  LLM Judge   │              │
│                                          │  (Quality)   │              │
│                                          └──────────────┘              │
│                                                 │                        │
│                                                 ▼                        │
│                                          ┌──────────────┐              │
│                                          │ NeMo Gym     │              │
│                                          │ Format       │              │
│                                          └──────────────┘              │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

## Part 1: Setup and Dependencies

First, let's install and import the necessary libraries.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install data-designer pydantic pandas

In [1]:
import json
import random
from typing import List, Optional
from pydantic import BaseModel, Field
import pandas as pd

# Data Designer imports
from data_designer.essentials import (
    ChatCompletionInferenceParams,
    DataDesigner,
    DataDesignerConfigBuilder,
    LLMStructuredColumnConfig,
    LLMTextColumnConfig,
    LocalFileSeedSource,
    ModelConfig,
    SamplingStrategy,
    ModelProvider
)

## Part 2: Load Tool Definitions

The **Workplace Assistant** environment has 26 tools across 5 databases:
- **Email**: Send, search, reply, forward, delete emails
- **Calendar**: Create, search, update, delete events
- **Analytics**: Query website visitor data and create plots
- **Project Management**: Manage tasks across Kanban boards
- **CRM**: Manage customer records and sales pipeline

These tools are designed to require **multi-step reasoning**. For example:
- "Email John about the meeting" → First lookup John's email, then send email
- "Reassign all of Sarah's leads to Mike" → Lookup emails, search customers, update each one

In [2]:
# Load tool definitions from separate JSON files (one per database)
import os

TOOLS_DIR = 'tools'

# Load environment config
with open(os.path.join(TOOLS_DIR, 'environment.json'), 'r') as f:
    env_config = json.load(f)

SYSTEM_PROMPT = env_config['system_prompt']
MULTI_STEP_PATTERNS = env_config['common_multi_step_patterns']

# Load tools from each database file
DATABASE_FILES = [
    'company_directory.json',
    'email.json', 
    'calendar.json',
    'analytics.json',
    'project_management.json',
    'customer_relationship_manager.json'
]

TOOLS = []
DATABASES = {}
TOOL_CATEGORIES = {}

for db_file in DATABASE_FILES:
    with open(os.path.join(TOOLS_DIR, db_file), 'r') as f:
        db_config = json.load(f)
        
    db_name = db_config['database']
    DATABASES[db_name] = {
        'description': db_config['description'],
        'data_schema': db_config['data_schema']
    }
    
    # Add tools and track category
    db_tools = db_config['tools']
    TOOLS.extend(db_tools)
    TOOL_CATEGORIES[db_name] = [t['name'] for t in db_tools]

print(f"Loaded {len(TOOLS)} tools across {len(DATABASES)} databases")
print(f"\nDatabases:")
for db_name, db_info in DATABASES.items():
    tool_count = len(TOOL_CATEGORIES[db_name])
    print(f"  - {db_name}: {tool_count} tools")
    print(f"    {db_info['description']}")

Loaded 27 tools across 6 databases

Databases:
  - company_directory: 1 tools
    Employee directory for looking up email addresses by name.
  - email: 6 tools
    Email inbox and outbox for sending, receiving, and managing emails.
  - calendar: 5 tools
    Calendar for managing meetings and events.
  - analytics: 6 tools
    Website analytics data for tracking visitor behavior and engagement.
  - project_management: 5 tools
    Project management board for tracking tasks across teams.
  - customer_relationship_manager: 4 tools
    CRM for managing customer records and sales pipeline.


In [3]:
# Helper function to format tools for prompts
def format_tools_for_prompt(tools: List[dict], include_schemas: bool = False) -> str:
    """Format tool definitions into a readable string for LLM prompts."""
    lines = []
    for tool in tools:
        lines.append(f"- **{tool['name']}**: {tool['description']}")
        if include_schemas:
            params = tool['parameters']['properties']
            if params:
                lines.append(f"  Parameters: {list(params.keys())}")
    return "\n".join(lines)

# Display tool summary by category
for category, tool_names in TOOL_CATEGORIES.items():
    print(f"\n### {category.upper()} ({len(tool_names)} tools)")
    category_tools = [t for t in TOOLS if t['name'] in tool_names]
    print(format_tools_for_prompt(category_tools))


### COMPANY_DIRECTORY (1 tools)
- **company_directory_find_email_address**: Finds all email addresses containing the given name (case-insensitive search).

### EMAIL (6 tools)
- **email_get_email_information_by_id**: Retrieves specific details of an email by its ID.
- **email_search_emails**: Searches for emails matching the given query across subject, body, or sender fields. The function matches an email if all words in the query appear in any of these fields.
- **email_send_email**: Sends an email to the specified recipient.
- **email_delete_email**: Deletes an email by its ID.
- **email_forward_email**: Forwards an email to the specified recipient.
- **email_reply_email**: Replies to an email by its ID.

### CALENDAR (5 tools)
- **calendar_get_event_information_by_id**: Returns the event for a given ID.
- **calendar_search_events**: Returns the events for a given query with pagination support.
- **calendar_create_event**: Creates a new event.
- **calendar_delete_event**: Deletes an

## Part 3: Define Output Schemas (Pydantic Models)

Data Designer uses Pydantic models to define structured output formats. This ensures the LLM generates data in a consistent, parseable format.

We define:
1. **ToolCall**: A single tool invocation with name and arguments
2. **AgentStep**: One step in a trajectory (thought + tool call + expected result)
3. **AgentTrajectory**: The complete multi-step solution

In [4]:
class ToolCall(BaseModel):
    """A single tool invocation."""
    name: str = Field(..., description="The name of the tool to call (e.g., 'email_send_email')")
    arguments: str = Field(..., description="JSON string of the tool arguments")


class AgentStep(BaseModel):
    """A single step in the agent's reasoning trajectory."""
    step_number: int = Field(..., description="The step number (1-indexed)")
    thought: str = Field(
        ..., 
        description="The agent's reasoning about what to do next and why. Should explain the purpose of the tool call."
    )
    tool_call: ToolCall = Field(..., description="The tool to call in this step")
    expected_result: str = Field(
        ..., 
        description="What information or state change we expect from this tool call"
    )


class AgentTrajectory(BaseModel):
    """Complete trajectory for solving a multi-step task."""
    reasoning_trace: List[AgentStep] = Field(
        ..., 
        description="The sequence of steps to solve the task. Should be 1-6 steps."
    )
    final_answer: str = Field(
        ..., 
        description="A brief confirmation of what was accomplished"
    )


class UserQueryJudgeScores(BaseModel):
    """Quality scores for the generated user query."""
    feasibility: int = Field(
        ..., ge=1, le=5, 
        description="Is the request achievable with the available tools? (1=impossible, 5=fully achievable)"
    )
    schema_compliance: int = Field(
        ..., ge=1, le=5, 
        description="Does the request use valid values as defined in tool schemas (e.g., valid board names, list names, statuses)? (1=uses invalid values, 5=all values valid)"
    )
    naturalness: int = Field(
        ..., ge=1, le=5, 
        description="Does the request sound like a natural user query? (1=robotic/artificial, 5=very natural)"
    )
    is_valid: bool = Field(
        ..., 
        description="True if the query is valid and should be kept, False if it should be discarded"
    )
    issues: str = Field(
        ..., 
        description="List any issues found (invalid enum values, impossible requests, etc.) or 'None' if valid"
    )


class TrajectoryJudgeScores(BaseModel):
    """Quality scores for the generated trajectory."""
    tool_validity: int = Field(
        ..., ge=1, le=5, 
        description="Are all tool names valid and arguments schema-compliant? (1=invalid tools/args, 5=all valid)"
    )
    argument_validity: int = Field(
        ..., ge=1, le=5, 
        description="Do all arguments use valid values as specified in tool descriptions? (1=invalid values, 5=all valid)"
    )
    completeness: int = Field(
        ..., ge=1, le=5, 
        description="Does the trajectory fully solve the user request? (1=incomplete, 5=fully complete)"
    )
    efficiency: int = Field(
        ..., ge=1, le=5, 
        description="Is the trajectory optimal without unnecessary steps? (1=very inefficient, 5=optimal)"
    )
    is_valid: bool = Field(
        ..., 
        description="True if the trajectory is valid and executable, False if it has errors"
    )
    issues: str = Field(
        ..., 
        description="List any issues found (invalid enum values, wrong tool names, missing steps, etc.) or 'None' if valid"
    )

## Part 4: Define Generation Prompts

The heart of synthetic data generation is the prompts. We need prompts for:

1. **User Query Generation**: Create realistic workplace requests
2. **Trajectory Simulation**: Generate the step-by-step solution
3. **Quality Judging**: Evaluate the generated data

### Key Principles:
- **Specificity**: Tell the LLM exactly what format you want
- **Examples**: Show don't tell - include concrete examples
- **Constraints**: Define what NOT to do (avoid trivial tasks, don't skip steps)

In [5]:
# Prompt 1: Generate a realistic user query that may require one or more tool calls
USER_QUERY_GENERATION_PROMPT = """
You are creating training data for a workplace assistant AI agent.

**Your Task:** Generate a realistic user request that requires the agent to use one or more tools to complete.

**Available Tools (with full schemas):**
{{ tools_json }}

**Selected Tool Category:** {{ category }}

**Multi-Step Pattern to Use:** {{ pattern }}

**CRITICAL - Valid Values:**
Many tool parameters have RESTRICTED VALUES specified in their descriptions. You MUST only reference values that exist in the tool schemas. Pay close attention to phrases like "One of:" in parameter descriptions.

Common restrictions to follow:
- `list_name`: Only use 'Backlog', 'In Progress', 'In Review', or 'Completed' (NOT 'Prospects', 'Todo', 'Pipeline', etc.)
- `board`: Only use 'Back end', 'Front end', or 'Design' (NOT 'Sales', 'Marketing', 'Engineering', etc.)
- `status`: Only use 'Qualified', 'Won', 'Lost', 'Lead', or 'Proposal' (NOT 'Active', 'Prospect', 'Closed', etc.)
- `product_interest`: Only use 'Software', 'Hardware', 'Services', 'Consulting', or 'Training'

**Guidelines:**
1. The request should sound natural - like something a real employee would ask
2. It should require 1-6 tool calls to complete
3. Include specific details that make the task concrete (names, dates, subjects)
4. Don't mention tool names or technical details - speak like a normal user
5. The task MUST be achievable with the available tools using ONLY valid parameter values
6. When referencing boards, lists, statuses, etc., use EXACTLY the values allowed in the tool schemas

**Examples by Complexity:**

*Simple (1 step):*
- "Reply to Carlos's last email about the prototype with 'Thanks, I'll review it tomorrow'"
- "Change the name of my 3pm meeting to 'Risk Management Forum'"
- "How many website visitors did we have last week?"

*Medium (2-3 steps):*
- "Send an email to John about the quarterly review meeting tomorrow"
- "Schedule a 30-minute sync with Lisa tomorrow at 2pm"
- "Get the total visits and engaged users for November"

*Complex (4-6 steps):*
- "Raj is taking over all of Akira's leads that are interested in software. Can you reassign them in the CRM?"
- "Forward the last email from marketing about the Q4 report to everyone on the design team"
- "Move all of Sarah's overdue tasks on the Back end board to the Backlog"

**Output:** Return ONLY the user request as a single string. No quotes, no explanation.
"""

print("User Query Generation Prompt loaded")

User Query Generation Prompt loaded


In [6]:
# Prompt 2: Simulate the agent's trajectory for solving the task
TRAJECTORY_SIMULATION_PROMPT = """
You are simulating an expert workplace assistant agent solving a task step-by-step.

**User Request:**
{{ user_query }}

**System Context:**
{{ system_prompt }}

**Available Tools:**
{{ tools_json }}

**Your Task:** Generate a step-by-step trajectory showing how the agent would solve this request.

**Guidelines:**
1. **Think Step-by-Step**: Each step should have a clear thought explaining WHY we're calling this tool
2. **Use Real Tool Names**: The tool_call.name must exactly match one of the available tools
3. **Valid JSON Arguments**: The tool_call.arguments must be valid JSON matching the tool's parameter schema
4. **Realistic IDs**: When referencing IDs discovered in previous steps, use placeholder format like "00000001"
5. **Complete the Task**: The trajectory must fully solve the user's request
6. **1-6 Steps**: Use the minimum number of steps needed. Simple tasks may need only 1 step.

**Common Patterns:**
- Look up a person's email before sending them a message
- Search for records before updating/deleting them
- Get information from one database to use in another
- Some tasks can be completed in a single step (e.g., reply to an email, update an event)

**Example Step:**
{% raw %}
```json
{
  "step_number": 1,
  "thought": "The user wants to email Raj, but I need his email address first. I'll look it up in the company directory.",
  "tool_call": {
    "name": "company_directory_find_email_address",
    "arguments": "{\"name\": \"Raj\"}"
  },
  "expected_result": "Raj's email address (likely raj.patel@atlas.com)"
}
```
{% endraw %}

Output the complete AgentTrajectory with all steps needed to solve the task.
"""

print("Trajectory Simulation Prompt loaded")

Trajectory Simulation Prompt loaded


In [7]:
# Prompt 3a: Judge the quality of the generated USER QUERY
USER_QUERY_JUDGE_PROMPT = """
You are a quality assurance judge evaluating a synthetically generated user query for training an AI workplace assistant.

**Generated User Query:**
{{ user_query }}

**Available Tools (with full schemas):**
{{ tools_json }}

**Your Task:** Evaluate whether this user query is valid and achievable with the available tools.

**CRITICAL - Check for Schema Compliance:**
Many tools have RESTRICTED VALUES for certain fields. The user query must only reference values that are valid according to the tool schemas. For example:
- If a tool says `list_name` must be one of 'Backlog', 'In Progress', 'In Review', 'Completed' - the query cannot ask for a "Prospects" list
- If a tool says `board` must be one of 'Back end', 'Front end', 'Design' - the query cannot ask for a "Sales" board  
- If a tool says `status` must be one of 'Qualified', 'Won', 'Lost', 'Lead', 'Proposal' - the query cannot use other statuses

**Evaluation Criteria:**

1. **Feasibility (1-5)**: Can this request be fulfilled using the available tools?
   - Score 1 if the request requires tools/capabilities that don't exist
   - Score 5 if the request is fully achievable with available tools

2. **Schema Compliance (1-5)**: Does the request use valid values?
   - Score 1 if the query references invalid enum values (wrong board names, list names, statuses, etc.)
   - Score 3 if the query is ambiguous but could map to valid values
   - Score 5 if all referenced values exactly match valid options in tool schemas

3. **Naturalness (1-5)**: Does this sound like a real user request?
   - Score 1 if robotic or artificial sounding
   - Score 5 if very natural and realistic

**is_valid:** Set to False if feasibility < 3 OR schema_compliance < 3. These queries should be discarded.

**issues:** List specific problems found. Examples:
- "References 'Sales' board but valid boards are: 'Back end', 'Front end', 'Design'"
- "References 'Prospects' list but valid lists are: 'Backlog', 'In Progress', 'In Review', 'Completed'"
- "None" if no issues found

**Output:** Return UserQueryJudgeScores with all fields.
"""

# Prompt 3b: Judge the quality of the generated TRAJECTORY
TRAJECTORY_JUDGE_PROMPT = """
You are a quality assurance judge evaluating a generated trajectory (sequence of tool calls) for training an AI workplace assistant.

**User Request:**
{{ user_query }}

**Generated Trajectory:**
{{ trajectory }}

**Available Tools (with full schemas):**
{{ tools_json }}

**Your Task:** Evaluate whether this trajectory correctly solves the user request using valid tool calls.

**CRITICAL - Check for Argument Validity:**
Tool arguments must use EXACTLY the values allowed by the tool schemas. For example:
- `list_name` must be one of: 'Backlog', 'In Progress', 'In Review', 'Completed' (NOT 'Prospects', 'Todo', etc.)
- `board` must be one of: 'Back end', 'Front end', 'Design' (NOT 'Sales', 'Marketing', etc.)
- `status` must be one of: 'Qualified', 'Won', 'Lost', 'Lead', 'Proposal' (NOT 'Active', 'Prospect', etc.)
- `product_interest` must be one of: 'Software', 'Hardware', 'Services', 'Consulting', 'Training'

**Evaluation Criteria:**

1. **Tool Validity (1-5)**: Are all tool names correct?
   - Score 1 if any tool name doesn't match available tools
   - Score 5 if all tool names exactly match

2. **Argument Validity (1-5)**: Are all arguments schema-compliant?
   - Score 1 if any argument uses invalid enum values or wrong types
   - Score 3 if arguments are mostly valid but some are ambiguous
   - Score 5 if all arguments perfectly match the schema requirements

3. **Completeness (1-5)**: Does the trajectory fully solve the request?
   - Score 1 if major parts of the request are unaddressed
   - Score 5 if the trajectory completely fulfills the request

4. **Efficiency (1-5)**: Is the trajectory optimal?
   - Score 1 if there are many unnecessary steps
   - Score 5 if the trajectory is optimal with no wasted steps

**is_valid:** Set to False if tool_validity < 4 OR argument_validity < 4. These trajectories have errors and should be discarded.

**issues:** List specific problems found. Examples:
- "Step 2 uses list_name='Prospects' but valid values are: 'Backlog', 'In Progress', 'In Review', 'Completed'"
- "Step 1 calls 'email_send' but correct tool name is 'email_send_email'"
- "None" if no issues found

**Output:** Return TrajectoryJudgeScores with all fields.
"""

print("User Query Judge Prompt loaded")
print("Trajectory Judge Prompt loaded")

User Query Judge Prompt loaded
Trajectory Judge Prompt loaded


## Part 5: Create Seed Data

Data Designer works by expanding seed data through LLM generation. Our seeds contain:
- Tool category to focus on
- Multi-step pattern to use
- Formatted tool descriptions

By varying the seeds, we ensure diversity in the generated data.

In [8]:
def create_seed_data(num_seeds: int = 100) -> pd.DataFrame:
    """
    Create seed data for the Data Designer pipeline.
    
    Each seed contains:
    - category: Which tool category to focus on
    - pattern: Which multi-step pattern to use
    - tools_description: Formatted tool descriptions
    - tools_json: Full tool schemas as JSON
    - system_prompt: The system context
    """
    seeds = []
    
    categories = list(TOOL_CATEGORIES.keys())
    patterns = [
        f"{p['pattern']}: {p['description']}" for p in MULTI_STEP_PATTERNS
    ]
    
    for i in range(num_seeds):
        # Select category and pattern (ensuring diversity)
        category = categories[i % len(categories)]
        pattern = patterns[i % len(patterns)]
        
        # Get tools for this category (plus company_directory for lookups)
        relevant_tool_names = TOOL_CATEGORIES[category] + TOOL_CATEGORIES.get('company_directory', [])
        relevant_tools = [t for t in TOOLS if t['name'] in relevant_tool_names]
        
        seeds.append({
            'seed_id': i,
            'category': category,
            'pattern': pattern,
            'tools_description': format_tools_for_prompt(relevant_tools, include_schemas=True),
            'tools_json': json.dumps(relevant_tools, indent=2),
            'tools_summary': format_tools_for_prompt(TOOLS),  # All tools for judge
            'system_prompt': SYSTEM_PROMPT,
        })
    
    return pd.DataFrame(seeds)

# Create seed data
seed_df = create_seed_data(num_seeds=50)
print(f"Created {len(seed_df)} seeds")
print(f"\nSample seed:")
print(seed_df.iloc[0].to_dict())

Created 50 seeds

Sample seed:
{'seed_id': 0, 'category': 'company_directory', 'pattern': "lookup_then_send_email: Look up a person's email address, then send them an email", 'tools_description': "- **company_directory_find_email_address**: Finds all email addresses containing the given name (case-insensitive search).\n  Parameters: ['name']", 'tools_json': '[\n  {\n    "type": "function",\n    "name": "company_directory_find_email_address",\n    "description": "Finds all email addresses containing the given name (case-insensitive search).",\n    "database": "company_directory",\n    "operation_type": "read",\n    "parameters": {\n      "type": "object",\n      "properties": {\n        "name": {\n          "type": "string",\n          "description": "Name or partial name to search for in email addresses"\n        }\n      },\n      "required": [],\n      "additionalProperties": false\n    },\n    "strict": false\n  }\n]', 'tools_summary': '- **company_directory_find_email_address**: Fi

In [9]:
# Save seeds to parquet for Data Designer
seed_df.to_parquet('workplace_assistant_seeds.parquet', index=False)
print("Seeds saved to workplace_assistant_seeds.parquet")

Seeds saved to workplace_assistant_seeds.parquet


## Part 6: Configure the Data Designer Pipeline

Now we wire everything together into a Data Designer workflow:

1. **Load Seeds** → Provides category, pattern, tools for each generation
2. **Generate User Query** → LLM creates realistic request
3. **Simulate Trajectory** → LLM generates step-by-step solution
4. **Judge Quality** → LLM evaluates validity, complexity, quality

The output is a dataset ready for NeMo Gym rollout collection.

In [10]:
import getpass

if "NVIDIA_API_KEY" not in os.environ or not os.environ["NVIDIA_API_KEY"]:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA API key: ")

In [ ]:
# Step 1: Define custom providers - pointing to local litellm proxy
LITELLM_PROXY_URL = "http://localhost:4000/v1"  # Adjust port if different

custom_providers = [
    ModelProvider(
        name="litellm-proxy",
        endpoint=LITELLM_PROXY_URL,
        provider_type="openai",  # litellm proxy is OpenAI-compatible
        api_key="not-needed",  # litellm proxy handles auth
    ),
]

# Model name must match the model_name in your litellm config
MODEL_ID = "gpt-120b"
MODEL_ALIAS = "gpt-120b"

model_configs = [
    ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider="litellm-proxy",
        inference_parameters=ChatCompletionInferenceParams(
            max_tokens=16384,
        ),
    )
]

# Step 3: Create DataDesigner with custom providers
data_designer = DataDesigner(model_providers=custom_providers)

# Step 4: Create config builder with custom models
config_builder = DataDesignerConfigBuilder(model_configs=model_configs)


In [12]:
def build_workplace_assistant_pipeline():
    """
    Build the complete Data Designer pipeline for generating 
    multi-step tool-calling training data.
    
    Pipeline stages:
    1. Generate user query
    2. Judge user query (filter invalid queries early)
    3. Generate trajectory 
    4. Judge trajectory (filter invalid trajectories)
    """
    
    # Initialize the config builder
    config_builder = DataDesignerConfigBuilder(model_configs=model_configs)
    
    # Load seed data
    seed_ref = LocalFileSeedSource(path='workplace_assistant_seeds.parquet')
    config_builder.with_seed_dataset(seed_ref, sampling_strategy=SamplingStrategy.SHUFFLE)
    
    # Column 1: Generate User Query
    # This creates a realistic workplace request based on the category and pattern
    config_builder.add_column(
        LLMTextColumnConfig(
            name="user_query",
            prompt=USER_QUERY_GENERATION_PROMPT,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 2: Judge User Query
    # Validates that the user query is feasible and uses valid enum values
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="user_query_judge",
            prompt=USER_QUERY_JUDGE_PROMPT,
            output_format=UserQueryJudgeScores,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 3: Simulate Agent Trajectory
    # This generates the step-by-step solution with tool calls
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="trajectory",
            prompt=TRAJECTORY_SIMULATION_PROMPT,
            output_format=AgentTrajectory,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 4: Judge Trajectory
    # Validates that the trajectory uses correct tool names and valid argument values
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="trajectory_judge",
            prompt=TRAJECTORY_JUDGE_PROMPT,
            output_format=TrajectoryJudgeScores,
            model_alias=MODEL_ALIAS,
        )
    )
    
    return config_builder

# Build the pipeline
pipeline = build_workplace_assistant_pipeline()
print("Pipeline configured with 4 generation columns:")
print("  1. user_query (text) - Generate realistic user request")
print("  2. user_query_judge (structured) - Validate query feasibility and schema compliance")
print("  3. trajectory (structured) - Generate step-by-step solution")
print("  4. trajectory_judge (structured) - Validate tool calls and argument values")

Pipeline configured with 4 generation columns:
  1. user_query (text) - Generate realistic user request
  2. user_query_judge (structured) - Validate query feasibility and schema compliance
  3. trajectory (structured) - Generate step-by-step solution
  4. trajectory_judge (structured) - Validate tool calls and argument values


In [13]:
data_designer.validate(pipeline)

[03:08:43] [INFO] ✅ Validation passed


## Part 7: Run the Pipeline (Local Testing)

For local testing, we'll run a small batch. For production, you'd use Big Iron to scale across GPUs.

In [14]:
preview = data_designer.preview(pipeline, num_records=2)

[03:08:54] [INFO] 🔁 Preview generation in progress
[03:08:54] [INFO] ✅ Validation passed
[03:08:54] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[03:08:54] [INFO] 🩺 Running health checks for models...
[03:08:54] [INFO]   |-- 👀 Checking 'gpt-120b' in provider named 'litellm-proxy' for model alias 'gpt-120b'...
[03:08:55] [INFO]   |-- ✅ Passed!
[03:08:55] [INFO] 🌱 Sampling 2 records from seed dataset
[03:08:55] [INFO]   |-- seed dataset size: 50 records
[03:08:55] [INFO]   |-- sampling strategy: shuffle
[03:08:55] [INFO] llm-text model configuration for generating column 'user_query'
[03:08:55] [INFO]   |-- model: 'gpt-120b'
[03:08:55] [INFO]   |-- model alias: 'gpt-120b'
[03:08:55] [INFO]   |-- model provider: 'litellm-proxy'
[03:08:55] [INFO]   |-- inference parameters: generation_type=chat-completion, max_parallel_requests=4, max_tokens=16384
[03:08:55] [INFO] 🐙 Processing llm-text column 'user_query' with 4 concurrent workers
[03:09:04] [INFO] llm-structured model c

In [84]:
preview.dataset

,seed_id,category,pattern,tools_description,tools_json,tools_summary,system_prompt,user_query,trajectory,judge_scores
0,40,project_management,compare_traffic_sources: Compare multiple traf...,- **company_directory_find_email_address**: Fi...,"[\n {\n ""type"": ""function"",\n ""name"": ""...",- **company_directory_find_email_address**: Fi...,"Today's date is Thursday, 2023-11-30 and the c...",Can you compare the tasks assigned to both Sar...,"{'reasoning_trace': [{'step_number': 1, 'thoug...","{'validity': 5, 'complexity': 4, 'quality': 4,..."
1,11,customer_relationship_manager,get_visitor_info_and_session_stats: Look up sp...,- **company_directory_find_email_address**: Fi...,"[\n {\n ""type"": ""function"",\n ""name"": ""...",- **company_directory_find_email_address**: Fi...,"Today's date is Thursday, 2023-11-30 and the c...",Can you show me all the customers currently as...,"{'reasoning_trace': [{'step_number': 1, 'thoug...","{'validity': 5, 'complexity': 4, 'quality': 4,..."


In [ ]:
# For local testing with a small batch
# 
# designer = data_designer.DataDesigner.from_config_builder(pipeline)
# results = designer.generate(num_records=5)
# results_df = results.to_pandas()
# print(results_df.head())

## Part 8: Convert to NeMo Gym Format

The generated data needs to be converted to the format expected by NeMo Gym for rollout collection. This format includes:

- `id`: Unique identifier
- `responses_create_params`: Input messages and tool schemas
- `ground_truth`: Expected tool calls
- `category`: Task category for stratification
- `environment_name`: Always "workplace_assistant"

In [15]:
def convert_to_nemo_gym_format(row: dict, idx: int) -> dict:
    """
    Convert a generated row to NeMo Gym rollout format.
    
    This format is what NeMo Gym's workplace_assistant environment expects.
    The rollout collector will:
    1. Send the input messages to the model
    2. Collect the model's tool calls
    3. Execute them in the environment
    4. Compare final state to ground_truth execution
    5. Compute reward (1 if states match, 0 otherwise)
    """
    
    # Parse the trajectory
    trajectory = row.get('trajectory', {})
    if isinstance(trajectory, str):
        trajectory = json.loads(trajectory)
    
    # Extract tool calls for ground truth
    ground_truth = []
    for step in trajectory.get('reasoning_trace', []):
        tool_call = step.get('tool_call', {})
        ground_truth.append({
            'name': tool_call.get('name', ''),
            'arguments': tool_call.get('arguments', '{}')
        })
    
    # Build the responses_create_params (OpenAI-compatible format)
    responses_create_params = {
        'input': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row.get('user_query', '')}
        ],
        'tools': TOOLS,  # All 27 tools available
        'parallel_tool_calls': False,  # Sequential execution
        'temperature': 1.0,  # For rollout diversity
    }
    
    return {
        'id': idx,
        'responses_create_params': responses_create_params,
        'ground_truth': ground_truth,
        'category': f"workplace_assistant_{row.get('category', 'general')}",
        'environment_name': 'workplace_assistant',
        # Metadata for analysis (not used by NeMo Gym)
        'user_query_judge': row.get('user_query_judge', {}),
        'trajectory_judge': row.get('trajectory_judge', {}),
        'pattern': row.get('pattern', ''),
    }

print("Conversion function defined")

Conversion function defined


In [16]:
def filter_high_quality(
    df: pd.DataFrame, 
    min_query_feasibility: int = 3,
    min_query_schema_compliance: int = 4,
    min_query_naturalness: int = 3,
    min_trajectory_tool_validity: int = 4,
    min_trajectory_argument_validity: int = 4,
    min_trajectory_completeness: int = 3,
    min_trajectory_efficiency: int = 3,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Filter generated data using DUAL-LEVEL quality control.
    
    This function applies TWO SEPARATE filtering stages:
    
    STAGE 1 - USER QUERY FILTERING:
    - Checks if the query is feasible with available tools
    - Validates schema compliance (correct enum values like board names, list names, statuses)
    - Ensures natural language quality
    
    STAGE 2 - TRAJECTORY FILTERING:
    - Validates all tool names are correct
    - Checks all arguments use valid enum values
    - Ensures the trajectory completely solves the request
    - Evaluates efficiency (no unnecessary steps)
    
    Args:
        df: DataFrame with user_query_judge and trajectory_judge columns
        min_query_feasibility: Minimum feasibility score (1-5)
        min_query_schema_compliance: Minimum schema compliance (1-5). CRITICAL for valid enum values.
        min_query_naturalness: Minimum naturalness score (1-5)
        min_trajectory_tool_validity: Minimum tool validity (1-5)
        min_trajectory_argument_validity: Minimum argument validity (1-5). CRITICAL for valid enum values.
        min_trajectory_completeness: Minimum completeness score (1-5)
        min_trajectory_efficiency: Minimum efficiency score (1-5)
        verbose: Print detailed statistics
    
    Returns:
        DataFrame with only high-quality examples that passed both filtering stages.
    """
    
    def parse_scores(scores):
        if isinstance(scores, str):
            return json.loads(scores)
        return scores or {}
    
    # Parse both judge scores
    df = df.copy()
    df['_query_scores'] = df['user_query_judge'].apply(parse_scores)
    df['_traj_scores'] = df['trajectory_judge'].apply(parse_scores)
    
    # =========================================================================
    # STAGE 1: USER QUERY LEVEL FILTERING
    # =========================================================================
    query_is_valid = df['_query_scores'].apply(lambda x: x.get('is_valid', False)) == True
    query_feasibility_ok = df['_query_scores'].apply(lambda x: x.get('feasibility', 0)) >= min_query_feasibility
    query_schema_ok = df['_query_scores'].apply(lambda x: x.get('schema_compliance', 0)) >= min_query_schema_compliance
    query_natural_ok = df['_query_scores'].apply(lambda x: x.get('naturalness', 0)) >= min_query_naturalness
    
    query_passed = query_is_valid & query_feasibility_ok & query_schema_ok & query_natural_ok
    
    # =========================================================================
    # STAGE 2: TRAJECTORY LEVEL FILTERING
    # =========================================================================
    traj_is_valid = df['_traj_scores'].apply(lambda x: x.get('is_valid', False)) == True
    traj_tool_ok = df['_traj_scores'].apply(lambda x: x.get('tool_validity', 0)) >= min_trajectory_tool_validity
    traj_args_ok = df['_traj_scores'].apply(lambda x: x.get('argument_validity', 0)) >= min_trajectory_argument_validity
    traj_complete_ok = df['_traj_scores'].apply(lambda x: x.get('completeness', 0)) >= min_trajectory_completeness
    traj_efficient_ok = df['_traj_scores'].apply(lambda x: x.get('efficiency', 0)) >= min_trajectory_efficiency
    
    traj_passed = traj_is_valid & traj_tool_ok & traj_args_ok & traj_complete_ok & traj_efficient_ok
    
    # =========================================================================
    # COMBINE BOTH STAGES
    # =========================================================================
    final_passed = query_passed & traj_passed
    
    if verbose:
        print("=" * 70)
        print("DUAL-LEVEL QUALITY FILTERING RESULTS")
        print("=" * 70)
        print(f"\nTotal records: {len(df)}")
        
        print(f"\n{'─'*70}")
        print("STAGE 1: USER QUERY FILTERING")
        print(f"{'─'*70}")
        print(f"  is_valid=True:           {query_is_valid.sum():4d} / {len(df)} ({query_is_valid.mean()*100:5.1f}%)")
        print(f"  feasibility >= {min_query_feasibility}:        {query_feasibility_ok.sum():4d} / {len(df)} ({query_feasibility_ok.mean()*100:5.1f}%)")
        print(f"  schema_compliance >= {min_query_schema_compliance}:  {query_schema_ok.sum():4d} / {len(df)} ({query_schema_ok.mean()*100:5.1f}%)")
        print(f"  naturalness >= {min_query_naturalness}:        {query_natural_ok.sum():4d} / {len(df)} ({query_natural_ok.mean()*100:5.1f}%)")
        print(f"  ──────────────────────────────────────────────────")
        print(f"  PASSED Stage 1:          {query_passed.sum():4d} / {len(df)} ({query_passed.mean()*100:5.1f}%)")
        
        print(f"\n{'─'*70}")
        print("STAGE 2: TRAJECTORY FILTERING")
        print(f"{'─'*70}")
        print(f"  is_valid=True:           {traj_is_valid.sum():4d} / {len(df)} ({traj_is_valid.mean()*100:5.1f}%)")
        print(f"  tool_validity >= {min_trajectory_tool_validity}:     {traj_tool_ok.sum():4d} / {len(df)} ({traj_tool_ok.mean()*100:5.1f}%)")
        print(f"  argument_validity >= {min_trajectory_argument_validity}: {traj_args_ok.sum():4d} / {len(df)} ({traj_args_ok.mean()*100:5.1f}%)")
        print(f"  completeness >= {min_trajectory_completeness}:      {traj_complete_ok.sum():4d} / {len(df)} ({traj_complete_ok.mean()*100:5.1f}%)")
        print(f"  efficiency >= {min_trajectory_efficiency}:        {traj_efficient_ok.sum():4d} / {len(df)} ({traj_efficient_ok.mean()*100:5.1f}%)")
        print(f"  ──────────────────────────────────────────────────")
        print(f"  PASSED Stage 2:          {traj_passed.sum():4d} / {len(df)} ({traj_passed.mean()*100:5.1f}%)")
        
        # Breakdown of rejections
        rejected_by_query_only = (~query_passed & traj_passed).sum()
        rejected_by_traj_only = (query_passed & ~traj_passed).sum()
        rejected_by_both = (~query_passed & ~traj_passed).sum()
        
        print(f"\n{'─'*70}")
        print("REJECTION BREAKDOWN")
        print(f"{'─'*70}")
        print(f"  Rejected by Query Judge only:      {rejected_by_query_only:4d}")
        print(f"  Rejected by Trajectory Judge only: {rejected_by_traj_only:4d}")
        print(f"  Rejected by BOTH judges:           {rejected_by_both:4d}")
        
        print(f"\n{'='*70}")
        print(f"FINAL RESULT: {final_passed.sum()} / {len(df)} passed ({final_passed.mean()*100:.1f}%)")
        print("=" * 70)
    
    filtered = df[final_passed].drop(columns=['_query_scores', '_traj_scores']).reset_index(drop=True)
    return filtered


def show_rejection_reasons(df: pd.DataFrame, num_examples: int = 5):
    """Show examples of rejected records with their issues for debugging at both levels."""
    
    def parse_scores(scores):
        if isinstance(scores, str):
            return json.loads(scores)
        return scores or {}
    
    df = df.copy()
    df['_query_scores'] = df['user_query_judge'].apply(parse_scores)
    df['_traj_scores'] = df['trajectory_judge'].apply(parse_scores)
    
    # =========================================================================
    # STAGE 1: USER QUERY REJECTIONS
    # =========================================================================
    query_invalid = df[df['_query_scores'].apply(lambda x: not x.get('is_valid', True))]
    query_schema_issues = df[df['_query_scores'].apply(lambda x: x.get('schema_compliance', 5) < 4)]
    
    print(f"\n{'='*70}")
    print("STAGE 1: USER QUERY ISSUES")
    print('='*70)
    
    if len(query_invalid) > 0:
        print(f"\n[INVALID QUERIES] ({len(query_invalid)} total)")
        for i, (_, row) in enumerate(query_invalid.head(num_examples).iterrows()):
            scores = row['_query_scores']
            print(f"\n  [{i+1}] Query: {row['user_query'][:80]}...")
            print(f"      Feasibility: {scores.get('feasibility', 'N/A')}/5 | Schema: {scores.get('schema_compliance', 'N/A')}/5")
            print(f"      Issues: {scores.get('issues', 'N/A')}")
    
    schema_only = query_schema_issues[~query_schema_issues.index.isin(query_invalid.index)]
    if len(schema_only) > 0:
        print(f"\n[SCHEMA COMPLIANCE ISSUES] ({len(schema_only)} additional)")
        for i, (_, row) in enumerate(schema_only.head(num_examples).iterrows()):
            scores = row['_query_scores']
            print(f"\n  [{i+1}] Query: {row['user_query'][:80]}...")
            print(f"      Schema Compliance: {scores.get('schema_compliance', 'N/A')}/5")
            print(f"      Issues: {scores.get('issues', 'N/A')}")
    
    if len(query_invalid) == 0 and len(schema_only) == 0:
        print("\n  No user query issues found!")
    
    # =========================================================================
    # STAGE 2: TRAJECTORY REJECTIONS
    # =========================================================================
    traj_invalid = df[df['_traj_scores'].apply(lambda x: not x.get('is_valid', True))]
    traj_arg_issues = df[df['_traj_scores'].apply(lambda x: x.get('argument_validity', 5) < 4)]
    
    print(f"\n{'='*70}")
    print("STAGE 2: TRAJECTORY ISSUES")
    print('='*70)
    
    if len(traj_invalid) > 0:
        print(f"\n[INVALID TRAJECTORIES] ({len(traj_invalid)} total)")
        for i, (_, row) in enumerate(traj_invalid.head(num_examples).iterrows()):
            scores = row['_traj_scores']
            print(f"\n  [{i+1}] Query: {row['user_query'][:80]}...")
            print(f"      Tool Validity: {scores.get('tool_validity', 'N/A')}/5 | Arg Validity: {scores.get('argument_validity', 'N/A')}/5")
            print(f"      Issues: {scores.get('issues', 'N/A')}")
    
    arg_only = traj_arg_issues[~traj_arg_issues.index.isin(traj_invalid.index)]
    if len(arg_only) > 0:
        print(f"\n[ARGUMENT VALIDITY ISSUES] ({len(arg_only)} additional)")
        for i, (_, row) in enumerate(arg_only.head(num_examples).iterrows()):
            scores = row['_traj_scores']
            print(f"\n  [{i+1}] Query: {row['user_query'][:80]}...")
            print(f"      Argument Validity: {scores.get('argument_validity', 'N/A')}/5")
            print(f"      Issues: {scores.get('issues', 'N/A')}")
    
    if len(traj_invalid) == 0 and len(arg_only) == 0:
        print("\n  No trajectory issues found!")
    
    print(f"\n{'='*70}\n")

print("Filter and debugging functions defined")

Filter and debugging functions defined


In [17]:
def save_for_nemo_gym(df: pd.DataFrame, output_path: str):
    """
    Save the dataset in JSONL format for NeMo Gym.
    
    Each line is a complete training example that can be used for:
    - Rollout collection (the model attempts to solve the task)
    - Reward computation (comparing model output to ground truth)
    - GRPO training (optimizing the policy using the rewards)
    """
    
    with open(output_path, 'w') as f:
        for idx, row in df.iterrows():
            record = convert_to_nemo_gym_format(row.to_dict(), idx)
            f.write(json.dumps(record) + '\n')
    
    print(f"Saved {len(df)} examples to {output_path}")

print("Save function defined")

Save function defined


## Part 9: End-to-End Workflow Summary

Here's the complete workflow for generating training data with **DUAL-LEVEL quality control**:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    DUAL-LEVEL QUALITY FILTERING                          │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  STAGE 1: USER QUERY LEVEL                                              │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ • Feasibility: Can the query be fulfilled with available tools?   │  │
│  │ • Schema Compliance: Does it use valid enum values?               │  │
│  │   (valid boards, list names, statuses, product interests)        │  │
│  │ • Naturalness: Does it sound like a real user request?           │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                              ↓                                           │
│  STAGE 2: TRAJECTORY LEVEL                                              │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ • Tool Validity: Are all tool names correct?                      │  │
│  │ • Argument Validity: Do all args use valid enum values?           │  │
│  │ • Completeness: Does the trajectory fully solve the request?     │  │
│  │ • Efficiency: Is it optimal without unnecessary steps?           │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                                                                          │
└─────────────────────────────────────────────────────────────────────────┘
```

```python
# 1. Create seeds
seed_df = create_seed_data(num_seeds=1000)
seed_df.to_parquet('seeds.parquet')

# 2. Build pipeline (4 generation stages)
pipeline = build_workplace_assistant_pipeline()
# Stages:
#   1. user_query - Generate realistic user request
#   2. user_query_judge - STAGE 1 quality validation
#   3. trajectory - Generate step-by-step solution
#   4. trajectory_judge - STAGE 2 quality validation

# 3. Run generation
results = data_designer.create(pipeline, num_records=1000)
results_df = results.load_dataset()

# 4. Debug - see why records are being rejected at each level
show_rejection_reasons(results_df, num_examples=5)

# 5. Apply DUAL-LEVEL filtering
filtered_df = filter_high_quality(
    results_df, 
    # Stage 1: User Query thresholds
    min_query_feasibility=3,
    min_query_schema_compliance=4,  # STRICT on enum values
    min_query_naturalness=3,
    # Stage 2: Trajectory thresholds
    min_trajectory_tool_validity=4,
    min_trajectory_argument_validity=4,  # STRICT on enum values
    min_trajectory_completeness=3,
    min_trajectory_efficiency=3,
)

# 6. Save for NeMo Gym
save_for_nemo_gym(filtered_df, 'workplace_assistant_train.jsonl')
```

**Why Dual-Level Filtering?**
- **Stage 1 (User Query)**: Catches queries like "add to Sales board" before wasting compute on trajectory generation
- **Stage 2 (Trajectory)**: Catches tool argument errors like `list_name: "Prospects"` that slipped through
- Both stages check schema compliance against the tool definitions

The output can then be used with NeMo Gym:

```bash
# Prepare data
ng_prepare_data "+config_paths=[workplace_assistant.yaml]" \
    +output_dirpath=data/workplace_assistant \
    +input_jsonl=workplace_assistant_train.jsonl

# Run GRPO training
python run_grpo_nemo_gym.py \
    --config=grpo_workplace_assistant.yaml \
    ++data.train_jsonl_fpath=data/workplace_assistant/train.jsonl
```

In [19]:
# Generate 10 examples using the pipeline
print("Generating 10 examples...")
results = data_designer.create(pipeline, num_records=10)

# Display the results
results_df = results.load_dataset()
print(f"\nGenerated {len(results_df)} records")
print("\nColumns:", list(results_df.columns))

# ============================================================================
# DUAL-LEVEL QUALITY FILTERING
# ============================================================================
# Stage 1: User Query Level - filters based on user_query_judge scores
# Stage 2: Trajectory Level - filters based on trajectory_judge scores
# ============================================================================

# First, show detailed rejection reasons for debugging
print("\n" + "="*70)
print("ANALYZING ISSUES AT BOTH LEVELS...")
show_rejection_reasons(results_df, num_examples=3)

# Apply dual-level filtering with strict schema compliance requirements
filtered_df = filter_high_quality(
    results_df,
    # Stage 1: User Query thresholds
    min_query_feasibility=3,
    min_query_schema_compliance=4,  # STRICT: Must use valid enum values
    min_query_naturalness=3,
    # Stage 2: Trajectory thresholds  
    min_trajectory_tool_validity=4,  # Must use correct tool names
    min_trajectory_argument_validity=4,  # STRICT: Must use valid enum values
    min_trajectory_completeness=3,
    min_trajectory_efficiency=3,
    verbose=True,
)

# Save to JSONL for NeMo Gym
def save_to_jsonl(df: pd.DataFrame, output_path: str):
    """Save filtered examples to JSONL in NeMo Gym format."""
    with open(output_path, 'w') as f:
        for idx, row in df.iterrows():
            record = convert_to_nemo_gym_format(row.to_dict(), idx)
            f.write(json.dumps(record) + '\n')
    print(f"\nSaved {len(df)} high-quality examples to {output_path}")

output_path = "workplace_assistant_train-gpt-oss.jsonl"
save_to_jsonl(filtered_df, output_path)

# Show a sample of the generated data
print("\n" + "="*50)
print("Sample generated data (passed both quality stages):")
filtered_df.head()

[03:11:18] [INFO] 🎨 Creating Data Designer dataset
[03:11:18] [INFO] ✅ Validation passed
[03:11:18] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[03:11:18] [INFO] 📂 Dataset path '/home/shashankv/bignlp/multistep-tool-rl-gym/DataDesigner/docs/colab_notebooks/workplace_assistant/artifacts/dataset' already exists. Dataset from this session
		     will be saved to '/home/shashankv/bignlp/multistep-tool-rl-gym/DataDesigner/docs/colab_notebooks/workplace_assistant/artifacts/dataset_01-29-2026_031118' instead.
[03:11:18] [INFO] 🩺 Running health checks for models...
[03:11:18] [INFO]   |-- 👀 Checking 'gpt-120b' in provider named 'litellm-proxy' for model alias 'gpt-120b'...


Generating 10 examples...


[03:11:18] [INFO]   |-- ✅ Passed!
[03:11:18] [INFO] ⏳ Processing batch 1 of 1
[03:11:18] [INFO] 🌱 Sampling 10 records from seed dataset
[03:11:18] [INFO]   |-- seed dataset size: 50 records
[03:11:18] [INFO]   |-- sampling strategy: shuffle
[03:11:18] [INFO] llm-text model configuration for generating column 'user_query'
[03:11:18] [INFO]   |-- model: 'gpt-120b'
[03:11:18] [INFO]   |-- model alias: 'gpt-120b'
[03:11:18] [INFO]   |-- model provider: 'litellm-proxy'
[03:11:18] [INFO]   |-- inference parameters: generation_type=chat-completion, max_parallel_requests=4, max_tokens=16384
[03:11:18] [INFO] 🐙 Processing llm-text column 'user_query' with 4 concurrent workers
[03:11:27] [INFO] llm-structured model configuration for generating column 'user_query_judge'
[03:11:27] [INFO]   |-- model: 'gpt-120b'
[03:11:27] [INFO]   |-- model alias: 'gpt-120b'
[03:11:27] [INFO]   |-- model provider: 'litellm-proxy'
[03:11:27] [INFO]   |-- inference parameters: generation_type=chat-completion, max_p


Generated 10 records

Columns: ['seed_id', 'category', 'pattern', 'tools_description', 'tools_json', 'tools_summary', 'system_prompt', 'user_query', 'user_query__reasoning_trace', 'user_query_judge', 'user_query_judge__reasoning_trace', 'trajectory', 'trajectory__reasoning_trace', 'trajectory_judge', 'trajectory_judge__reasoning_trace']

ANALYZING ISSUES AT BOTH LEVELS...

STAGE 1: USER QUERY ISSUES

[INVALID QUERIES] (4 total)

  [1] Query: Please locate the email address of the customer Jane Smith, who showed interest ...
      Feasibility: 2/5 | Schema: 5/5
      Issues: The request asks to forward the customer's most recent email, but no available tool can retrieve or send email content. Only email address lookup is supported.

  [2] Query: Please find the email address for Michael Chen and schedule a 30‑minute meeting ...
      Feasibility: 2/5 | Schema: 5/5
      Issues: No available tool to schedule a meeting; the request requires calendar/scheduling functionality which is not 

,seed_id,category,pattern,tools_description,tools_json,tools_summary,system_prompt,user_query,user_query__reasoning_trace,user_query_judge,user_query_judge__reasoning_trace,trajectory,trajectory__reasoning_trace,trajectory_judge,trajectory_judge__reasoning_trace
0,5,customer_relationship_manager,lookup_then_create_event: Look up a person's e...,- **company_directory_find_email_address**: Fi...,"[ { ""type"": ""function"", ""name"": ""com...",- **company_directory_find_email_address**: Fi...,"Today's date is Thursday, 2026-01-29 and the c...",Please look up Michael Thompson's email in the...,We need to produce a realistic user request th...,"{'feasibility': 5, 'is_valid': True, 'issues':...","We need to evaluate the user query: ""Please lo...",{'final_answer': 'Michael Thompson has been ad...,We need to produce trajectory steps. Task: Loo...,"{'argument_validity': 5, 'completeness': 5, 'e...",We need to evaluate the trajectory. User requ...
1,41,customer_relationship_manager,lookup_then_create_task: Look up a person's em...,- **company_directory_find_email_address**: Fi...,"[ { ""type"": ""function"", ""name"": ""com...",- **company_directory_find_email_address**: Fi...,"Today's date is Thursday, 2026-01-29 and the c...",Can you look up Maya Patel's email and then cr...,We need to generate a realistic user request t...,"{'feasibility': 5, 'is_valid': True, 'issues':...","We need to evaluate the user query: ""Can you l...","{'final_answer': ""The lead for Acme Corp has b...",We need to produce trajectory steps. Task: loo...,"{'argument_validity': 5, 'completeness': 5, 'e...",We need to evaluate the generated trajectory. ...
2,21,analytics,search_then_update_status: Search for customer...,- **company_directory_find_email_address**: Fi...,"[ { ""type"": ""function"", ""name"": ""com...",- **company_directory_find_email_address**: Fi...,"Today's date is Thursday, 2026-01-29 and the c...",I need a quick summary of our website performa...,We need to generate a realistic user request t...,"{'feasibility': 5, 'is_valid': True, 'issues':...","We need to evaluate the user query: ""I need a ...",{'final_answer': 'The summary for 2023-10-01 t...,We need to produce a trajectory of tool calls ...,"{'argument_validity': 5, 'completeness': 5, 'e...",We need to evaluate the given trajectory. Use...


## Next Steps

1. **Customize prompts** for your specific domain/tools
2. **Add more patterns** to increase trajectory diversity
3. **Tune judge thresholds** based on manual inspection
4. **Iterate on quality** - check failed examples, improve prompts
5. **Scale up**